# Zero-Shot Prompting with LangChain/OpenAI

#### Step 1: Set up the OpenAI API Key
* The code imports the necessary libraries.
* The os is used for interacting with the operating system, and openai is the library required to work with OpenAI's API.
* If .env file is setup in the project, then os.getenv can be used to access the api_key

In [1]:
import os
from pathlib import Path

import openai
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

In [2]:
deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

#### Step 2: Define a Function to Get Completion

The `get_completion` function is responsible for sending a prompt to the OpenAI model and receiving its response.

Parameters:

* prompt: It is the text input for which the model will generate a completion.
* model: The gpt-x (whichever is deployed at the endpoint) model is used to perform the tasks.
The `client.chat.completions.create` function is used to send a request to the API endpoint.

This request includes the model, the input messages (formatted as a list of dictionaries with user roles and content), and a temperature setting.

In [3]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI.

    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.

    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1, # makes the model much less random and more likely to pick the highest-probability next token.
            top_p=0.8, # restricts sampling to the smallest set of tokens whose cumulative probability is at least 20%, so only the most likely options are considered.
            max_tokens=50
        )

        return response.model_dump()  # Return the full response as dict

    except Exception as e:
        return {"error": str(e)}

**Role** --> Defining who is speaking

    System : Sets the behaviour of the model
    User : Who inputs the data or instruction
    Assistant : The models's Response
    
Content : The actual text or instruction which is passed by us.

#### Step 3: Define Your Prompt
* The prompt variable is defined with a simple translation task.

In [4]:
prompt = "Explain human intelligence with respect to cognitive ability. \
Give me Reasoning behind the response as Reasoning: \
and give me citations as citations: "
response = get_completion(prompt)

In [5]:
type(response)

dict

In [6]:
response.keys()

dict_keys(['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage', 'prompt_filter_results'])

In [7]:
# Ensure it's a dictionary
if isinstance(response, dict) and "choices" in response:
    print(response["choices"][0]["message"]["content"])
else:
    print("Error:", response)

Human intelligence, with respect to cognitive ability, refers to the capacity of individuals to acquire knowledge, understand complex ideas, adapt effectively to the environment, solve problems, and use reasoning in various contexts. Cognitive abilities encompass a range of mental processes including memory,


In [11]:
response['choices'][0]['message']

{'content': 'Human intelligence, with respect to cognitive ability, refers to the capacity of individuals to acquire knowledge, understand complex ideas, adapt effectively to the environment, solve problems, and use reasoning in various contexts. Cognitive abilities encompass a range of mental processes including memory,',
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [ ]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI with a preloaded conversation.
    """
    messages = [
        {"role": "system", "content": "You are a math tutor"},
        {"role": "user", "content": "Explain me the ..."},
        {"role": "assistant", "content": "The concept is like..."},
        {"role": "user", "content": prompt},   # <-- use the function prompt here
    ]

    try:
        response = client.chat.completions.create(
            model=deployment_name,   # <-- deployment name, not raw model name
            messages=messages,
            temperature=0.9,
            top_p=0.8,
            max_tokens=50

        )
        #return response.model_dump()
        return response.choices[0].message.content   # <-- just return the text
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
reply = get_completion("Give me an example of probability in daily life")
print(reply)

Sure! Here's an example of probability in daily life:

Imagine you have a bag with 5 red apples and 3 green apples. If you randomly pick one apple from the bag without looking, the probability of picking a red apple is the number of red apples divided by the total number of apples. 

So, the probability = Number of red apples / Total number of apples = 5 / (5 + 3) = 5/8.

This means there is a 5 out of 8 chance, or about 62.5%, that you will pick a red apple. 

Probability helps us understand how likely an event is to happen in everyday situations!


In [13]:
prompt = "explain trignometry"

response = get_completion(prompt)
print(response)

Sure! Trigonometry is a branch of mathematics that studies the relationships between the angles and sides of triangles, especially right-angled triangles.

Here are the key points:

1. **Basic Ratios**: In a right triangle, the primary trigonometric ratios are:
   - **Sine (sin)** = Opposite side / Hypotenuse
   - **Cosine (cos)** = Adjacent side / Hypotenuse
   - **Tangent (tan)** = Opposite side / Adjacent side

2. **Uses**: Trigonometry helps you find unknown sides or angles in triangles, which is useful in fields like physics, engineering, architecture, and navigation.

3. **Unit Circle**: Trigonometry also extends beyond triangles using the unit circle, where angles correspond to points on a circle with radius 1, allowing sine and cosine to be defined for all angles.

4. **Other Functions**: There are also cosecant (csc), secant (sec), and cotangent (cot), which are reciprocals of sine, cosine, and tangent respectively.

If you'd like, I can explain any of these points in more det

In [15]:
import langchain
print(langchain.__version__)
from langchain_openai import AzureChatOpenAI

1.2.15


In [22]:
client_lc = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    temperature=0.5,
    max_tokens=50
)

In [23]:
def get_completion_lc(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

In [24]:
prompt = "Explain human intelligence with respect to cognitive ability"
response = get_completion_lc(prompt)

print(response)

Human intelligence, in relation to cognitive ability, refers to the mental capacities that enable individuals to acquire knowledge, understand complex ideas, adapt effectively to their environment, solve problems, and use reasoning and judgment. Cognitive abilities are the underlying mental processes that support these
